In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Load the CSV file

In [ ]:
import pandas as pd
import os

# Define the paths for the complete DataFrame and the original CSV
final_output_image_path = '/content/drive/MyDrive/Tesis/books_with_all_embeddings.parquet'
text_embeddings_path = '/content/drive/MyDrive/Tesis/books_with_embeddings.parquet' # Path for DataFrame with text embeddings only
original_file_path = '/content/drive/MyDrive/Tesis/books_for_sentence_bert.csv'

df = None # Initialize df

# 1. Check if the DataFrame with all embeddings already exists
if os.path.exists(final_output_image_path):
    print(f"Loading DataFrame with all embeddings from {final_output_image_path}")
    df = pd.read_parquet(final_output_image_path)
    print("DataFrame loaded successfully with existing text and image embeddings.")
# 2. If not, check if the DataFrame with only text embeddings exists
elif os.path.exists(text_embeddings_path):
    print(f"'{final_output_image_path}' not found. Loading DataFrame with text embeddings from {text_embeddings_path}")
    df = pd.read_parquet(text_embeddings_path)
    print("DataFrame loaded successfully with existing text embeddings. Image embeddings will be generated.")
# 3. If neither exists, load the original CSV
else:
    print(f"Neither '{final_output_image_path}' nor '{text_embeddings_path}' found. Loading original CSV from {original_file_path}")
    df = pd.read_csv(original_file_path)
    print("Original DataFrame loaded successfully. Both text and image embeddings will be generated.")

print("First 5 rows of the loaded DataFrame:")
display(df.head())

'/content/drive/MyDrive/Tesis/books_with_all_embeddings.parquet' not found. Loading DataFrame with text embeddings from /content/drive/MyDrive/Tesis/books_with_embeddings.parquet
DataFrame loaded successfully with existing text embeddings. Image embeddings will be generated.
First 5 rows of the loaded DataFrame:


,id,book_id,titulo,authors,image_url,genres,textual_column,embeddings
0,1,2767052,The Hunger Games,Suzanne Collins,http://books.google.com/books/content?id=sJdUA...,"['favorites', 'currently-reading', 'young-adul...",Title: The Hunger Games. Author: Suzanne Colli...,"[0.036066729575395584, 0.03727075085043907, -0..."
1,4,2657,To Kill a Mockingbird,Harper Lee,http://books.google.com/books/content?id=ncuX8...,"['classics', 'favorites', 'to-read', 'classic'...",Title: To Kill a Mockingbird. Author: Harper L...,"[0.044271551072597504, 0.03307667374610901, -0..."
2,5,4671,The Great Gatsby,F. Scott Fitzgerald,http://books.google.com/books/content?id=fIlQD...,"['classics', 'favorites', 'fiction', 'classic'...",Title: The Great Gatsby. Author: F. Scott Fitz...,"[0.01517120935022831, 0.029992535710334778, -0..."
3,6,11870085,The Fault in Our Stars,John Green,http://books.google.com/books/content?id=Dc2LD...,"['favorites', 'to-read', 'young-adult', 'ficti...",Title: The Fault in Our Stars. Author: John Gr...,"[0.020600659772753716, 0.007697263732552528, -..."
4,7,5907,The Hobbit or There and Back Again,J.R.R. Tolkien,http://books.google.com/books/content?id=ljWL5...,"['fantasy', 'favorites', 'classics', 'to-read'...",Title: The Hobbit or There and Back Again. Aut...,"[0.045859359204769135, 0.01891307905316353, -0..."


### Build the textual column for Sentence-BERT

In [ ]:
# Create the 'textual_column' using the specified template
df['textual_column'] = df.apply(lambda row: f"Title: {row['titulo']}. Author: {row['authors']}. Genres: {row['genres']}.", axis=1)

print("DataFrame with the new 'textual_column':")
display(df[['titulo', 'authors', 'genres', 'textual_column']].head())

DataFrame with the new 'textual_column':


,titulo,authors,genres,textual_column
0,The Hunger Games,Suzanne Collins,"['favorites', 'currently-reading', 'young-adul...",Title: The Hunger Games. Author: Suzanne Colli...
1,To Kill a Mockingbird,Harper Lee,"['classics', 'favorites', 'to-read', 'classic'...",Title: To Kill a Mockingbird. Author: Harper L...
2,The Great Gatsby,F. Scott Fitzgerald,"['classics', 'favorites', 'fiction', 'classic'...",Title: The Great Gatsby. Author: F. Scott Fitz...
3,The Fault in Our Stars,John Green,"['favorites', 'to-read', 'young-adult', 'ficti...",Title: The Fault in Our Stars. Author: John Gr...
4,The Hobbit or There and Back Again,J.R.R. Tolkien,"['fantasy', 'favorites', 'classics', 'to-read'...",Title: The Hobbit or There and Back Again. Aut...


### Generate Sentence-BERT Embeddings

In [ ]:
# Install the sentence-transformers library if you haven't already
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 12.9 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.0
    Uninstalling sentence-transformers-5.4.0:
      Successfully uninstalled sentence-transformers-5.4.0


In [ ]:
from sentence_transformers import SentenceTransformer
import torch # Import torch to check for CUDA

# Load a pre-trained Sentence-BERT model
# 'paraphrase-MiniLM-L6-v2' is a good balance of speed and performance
# You can choose other models like 'all-MiniLM-L6-v2' or 'all-mpnet-base-v2' if needed
model_name = 'sentence-transformers/all-mpnet-base-v2'
# Determine the device to use (GPU if available, else CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model = SentenceTransformer(model_name, device=device)

print(f"Sentence-BERT model '{model_name}' loaded successfully on {device}.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT model 'sentence-transformers/all-mpnet-base-v2' loaded successfully on cuda.


In [ ]:
import numpy as np
import os

# Define chunk size for processing and checkpointing frequency
chunk_size = 1000 # Process 1000 texts at a time before saving a checkpoint
checkpoint_file = '/content/drive/MyDrive/Tesis/books_embeddings_checkpoint.npy' # Path to save intermediate embeddings
final_output_path = '/content/drive/MyDrive/Tesis/books_with_embeddings.parquet' # Path to save final DataFrame

# Ensure to handle potential NaN values by converting them to empty strings
df['textual_column'] = df['textual_column'].fillna('')
texts_to_encode = df['textual_column'].tolist()
total_texts = len(texts_to_encode)

all_embeddings = []
start_index = 0

# Check if a checkpoint file exists to resume
if os.path.exists(checkpoint_file):
    print(f"Resuming from checkpoint file: {checkpoint_file}")
    # Use allow_pickle=True for loading arrays that contain Python objects (like lists)
    loaded_embeddings = np.load(checkpoint_file, allow_pickle=True)
    all_embeddings.extend(loaded_embeddings.tolist()) # Convert back to list of lists
    start_index = len(all_embeddings)
    print(f"Resumed from index {start_index}/{total_texts}")

# Set batch_size for model.encode. This is for internal parallelism within each chunk.
batch_size = 64

# Iterate through the texts in chunks for processing and checkpointing
for i in range(start_index, total_texts, chunk_size):
    end_index = min(i + chunk_size, total_texts)
    current_chunk = texts_to_encode[i:end_index]

    print(f"Processing texts from index {i} to {end_index-1}...")
    # Generate embeddings for the current chunk
    chunk_embeddings = model.encode(current_chunk, show_progress_bar=False, batch_size=batch_size)
    all_embeddings.extend(chunk_embeddings.tolist())

    # Save checkpoint periodically
    # Saving as object array to handle potentially varied embedding shapes if model changes (though unlikely here)
    np.save(checkpoint_file, np.array(all_embeddings, dtype=object))
    print(f"Checkpoint saved. Current progress: {end_index}/{total_texts}")

# Convert list of embeddings to a final NumPy array
embeddings_array = np.array(all_embeddings)

# Add embeddings to the DataFrame as a list of arrays (or numpy arrays)
df['embeddings'] = list(embeddings_array)

print("\nEmbeddings generated successfully.")
print(f"Shape of embeddings: {embeddings_array.shape}")
print("First 5 embeddings (first 10 dimensions):")
display(pd.DataFrame(embeddings_array[:5, :10]))

# Save the entire DataFrame with embeddings to a Parquet file for efficient storage
df.to_parquet(final_output_path, index=False)
print(f"Final DataFrame with embeddings saved to {final_output_path}")

Processing texts from index 0 to 999...
Checkpoint saved. Current progress: 1000/6585
Processing texts from index 1000 to 1999...
Checkpoint saved. Current progress: 2000/6585
Processing texts from index 2000 to 2999...
Checkpoint saved. Current progress: 3000/6585
Processing texts from index 3000 to 3999...
Checkpoint saved. Current progress: 4000/6585
Processing texts from index 4000 to 4999...
Checkpoint saved. Current progress: 5000/6585
Processing texts from index 5000 to 5999...
Checkpoint saved. Current progress: 6000/6585
Processing texts from index 6000 to 6584...
Checkpoint saved. Current progress: 6585/6585

Embeddings generated successfully.
Shape of embeddings: (6585, 768)
First 5 embeddings (first 10 dimensions):


,0,1,2,3,4,5,6,7,8,9
0,0.036067,0.037271,-0.020196,0.022858,0.010336,0.031754,-0.003490,-0.041021,0.047956,-0.012575
1,0.044272,0.033077,-0.020180,0.009792,-0.009760,0.051099,0.023757,0.024269,0.007475,-0.033793
2,0.015171,0.029993,-0.024497,0.029453,-0.007605,0.040373,0.001013,0.019199,0.039081,-0.011965
3,0.020601,0.007697,-0.014884,-0.016340,-0.017072,0.056456,0.015615,0.024740,0.039245,0.014261
4,0.045859,0.018913,-0.052938,0.034684,-0.040848,0.029240,0.024942,0.000696,-0.000833,-0.038076


Final DataFrame with embeddings saved to /content/drive/MyDrive/Tesis/books_with_embeddings.parquet


### Generate OpenCLIP Image Embeddings

In [ ]:
# Install necessary libraries for OpenCLIP and image processing
!pip install -qU open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
import requests
from io import BytesIO
import numpy as np
import os

# Load a pre-trained OpenCLIP model
# You can choose other models, e.g., 'ViT-L-14', 'ViT-g-14' depending on performance needs
model_name_clip = "ViT-B-32"
pretrained = "openai" # or "laion2b_s34b_b79k" etc.

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

clip_model, _, preprocess = open_clip.create_model_and_transforms(model_name_clip, pretrained=pretrained, device=device)
clip_model.eval() # Set model to evaluation mode

print(f"OpenCLIP model '{model_name_clip}' loaded successfully on {device}.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


OpenCLIP model 'ViT-B-32' loaded successfully on cuda.


In [ ]:
# Define paths for image embeddings checkpoint and final output
image_checkpoint_file = '/content/drive/MyDrive/Tesis/books_image_embeddings_checkpoint.npy'
final_output_image_path = '/content/drive/MyDrive/Tesis/books_with_all_embeddings.parquet' # Final DataFrame with all embeddings

image_urls_to_encode = df['image_url'].tolist()
total_images = len(image_urls_to_encode)

all_image_embeddings = []
start_image_index = 0

# Check for existing image embeddings checkpoint to resume
if os.path.exists(image_checkpoint_file):
    print(f"Resuming image embedding from checkpoint: {image_checkpoint_file}")
    loaded_image_embeddings = np.load(image_checkpoint_file, allow_pickle=True)
    all_image_embeddings.extend(loaded_image_embeddings.tolist())
    start_image_index = len(all_image_embeddings)
    print(f"Resumed from image index {start_image_index}/{total_images}")

# Function to load and preprocess image from URL
def load_and_preprocess_image(url):
    if pd.isna(url) or not isinstance(url, str):
        return None
    try:
        response = requests.get(url, timeout=10) # Add timeout for robustness
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return preprocess(image)
    except requests.exceptions.RequestException as e:
        print(f"Error downloading image from {url}: {e}")
        return None
    except Exception as e:
        print(f"Error processing image from {url}: {e}")
        return None

chunk_size_images = 100 # Process 100 images at a time for checkpointing
image_batch_size = 32 # Batch size for clip_model.encode_image for internal parallelism

for i in range(start_image_index, total_images, chunk_size_images):
    end_index_images = min(i + chunk_size_images, total_images)
    current_image_urls = image_urls_to_encode[i:end_index_images]

    print(f"Processing images from index {i} to {end_index_images-1}...")

    processed_images = []
    valid_indices = []
    for j, url in enumerate(current_image_urls):
        processed_img = load_and_preprocess_image(url)
        if processed_img is not None:
            processed_images.append(processed_img)
            valid_indices.append(j) # Store original index within the chunk

    if processed_images:
        processed_images_tensor = torch.stack(processed_images).to(device)
        with torch.no_grad():
            chunk_image_embeddings = clip_model.encode_image(processed_images_tensor).cpu().numpy()

        # Create a placeholder for the full chunk embeddings (including None for failed downloads)
        full_chunk_embeddings_placeholder = [None] * len(current_image_urls)
        for k, original_idx in enumerate(valid_indices):
            full_chunk_embeddings_placeholder[original_idx] = chunk_image_embeddings[k].tolist() # Store as list

        all_image_embeddings.extend(full_chunk_embeddings_placeholder)
    else:
        # If no images were processed in the chunk, extend with None for all
        all_image_embeddings.extend([None] * len(current_image_urls))

    # Save checkpoint periodically
    np.save(image_checkpoint_file, np.array(all_image_embeddings, dtype=object))
    print(f"Image embedding checkpoint saved. Current progress: {end_index_images}/{total_images}")

# Convert list of image embeddings to a final NumPy array, handling potential None values
# First, find the dimension of a valid embedding (if any) to fill None values consistently
sample_embedding_dim = None
for emb in all_image_embeddings:
    if emb is not None:
        sample_embedding_dim = len(emb)
        break

if sample_embedding_dim is not None:
    # Replace None values with an array of NaNs of the correct dimension
    image_embeddings_array = np.array([
        emb if emb is not None else np.full(sample_embedding_dim, np.nan).tolist()
        for emb in all_image_embeddings
    ], dtype=object)
    print(f"Shape of image embeddings: {image_embeddings_array.shape}")
    print("First 5 image embeddings (first 10 dimensions):")
    display(pd.DataFrame(np.stack(image_embeddings_array[:5, :10] if image_embeddings_array.size > 0 else np.array([]))))
else:
    image_embeddings_array = np.array([])
    print("No valid image embeddings were generated.")

# Add image embeddings to the DataFrame
df['image_embeddings'] = list(image_embeddings_array)

print("\nImage embeddings generated successfully.")
print("DataFrame with image embeddings:")
display(df[['image_url', 'image_embeddings']].head())

# Save the entire DataFrame with both text and image embeddings to a Parquet file
df.to_parquet(final_output_image_path, index=False)
print(f"Final DataFrame with all embeddings saved to {final_output_image_path}")

Processing images from index 0 to 99...
Image embedding checkpoint saved. Current progress: 100/6585
Processing images from index 100 to 199...
Image embedding checkpoint saved. Current progress: 200/6585
Processing images from index 200 to 299...
Image embedding checkpoint saved. Current progress: 300/6585
Processing images from index 300 to 399...
Image embedding checkpoint saved. Current progress: 400/6585
Processing images from index 400 to 499...
Image embedding checkpoint saved. Current progress: 500/6585
Processing images from index 500 to 599...
Image embedding checkpoint saved. Current progress: 600/6585
Processing images from index 600 to 699...
Image embedding checkpoint saved. Current progress: 700/6585
Processing images from index 700 to 799...
Image embedding checkpoint saved. Current progress: 800/6585
Processing images from index 800 to 899...
Image embedding checkpoint saved. Current progress: 900/6585
Processing images from index 900 to 999...
Image embedding checkpoi

,0,1,2,3,4,5,6,7,8,9
0,0.275819,0.271356,-0.127995,-0.321001,0.701547,0.089646,0.346398,-0.12973,-0.148652,0.125662
1,0.439297,-0.321503,0.18169,-0.32066,0.292497,-0.043947,-0.191819,0.319021,-0.053612,0.583425
2,0.389695,-0.219626,-0.686113,-0.191287,0.574115,-0.13602,-0.180437,-0.679626,0.305644,0.33933
3,0.13765,-0.411139,-0.37617,0.149913,0.180338,-0.223009,-0.12954,-1.013744,-0.08539,0.555519
4,0.078271,0.330439,0.422956,-0.060404,0.105241,-0.719519,-0.101536,0.258414,-0.570796,0.25761



Image embeddings generated successfully.
DataFrame with image embeddings:


,image_url,image_embeddings
0,http://books.google.com/books/content?id=sJdUA...,"[0.2758193612098694, 0.2713555097579956, -0.12..."
1,http://books.google.com/books/content?id=ncuX8...,"[0.4392969012260437, -0.32150304317474365, 0.1..."
2,http://books.google.com/books/content?id=fIlQD...,"[0.38969525694847107, -0.2196260690689087, -0...."
3,http://books.google.com/books/content?id=Dc2LD...,"[0.1376495063304901, -0.4111386835575104, -0.3..."
4,http://books.google.com/books/content?id=ljWL5...,"[0.07827101647853851, 0.3304389417171478, 0.42..."


Final DataFrame with all embeddings saved to /content/drive/MyDrive/Tesis/books_with_all_embeddings.parquet


In [15]:
import numpy as np
from sklearn.preprocessing import normalize

# Helper function to safely normalize an embedding
def safe_normalize(embedding):
    if embedding is None:
        return None
    try:
        # Convert to numpy array with float32 dtype
        np_embedding = np.array(embedding, dtype=np.float32)
        # Check if the array contains any NaN values
        if np.any(np.isnan(np_embedding)):
            return None # Filter out embeddings with NaNs
        # Normalize and convert back to list
        normalized = normalize(np_embedding.reshape(1, -1))[0].tolist()
        return normalized
    except (ValueError, TypeError):
        # Handle cases where conversion to numpy array or normalization fails
        return None

print("Normalizing text embeddings...")
df['normalized_embeddings'] = df['embeddings'].apply(safe_normalize)
print("Text embeddings normalized successfully, filtering invalid embeddings.")

print("Normalizing image embeddings...")
df['normalized_image_embeddings'] = df['image_embeddings'].apply(safe_normalize)
print("Image embeddings normalized successfully, filtering invalid embeddings.")

print("DataFrame with new normalized embeddings:")
display(df[['titulo', 'normalized_embeddings', 'normalized_image_embeddings']].head())

Normalizing text embeddings...
Text embeddings normalized successfully, filtering invalid embeddings.
Normalizing image embeddings...
Image embeddings normalized successfully, filtering invalid embeddings.
DataFrame with new normalized embeddings:


,titulo,normalized_embeddings,normalized_image_embeddings
0,The Hunger Games,"[0.036066725850105286, 0.03727074712514877, -0...","[0.026253582909703255, 0.02582869678735733, -0..."
1,To Kill a Mockingbird,"[0.044271551072597504, 0.03307667374610901, -0...","[0.043771255761384964, -0.03203435242176056, 0..."
2,The Great Gatsby,"[0.01517120935022831, 0.029992535710334778, -0...","[0.041459228843450546, -0.023365763947367668, ..."
3,The Fault in Our Stars,"[0.020600661635398865, 0.007697264663875103, -...","[0.014725574292242527, -0.043983109295368195, ..."
4,The Hobbit or There and Back Again,"[0.04585936293005943, 0.018913080915808678, -0...","[0.0081606050953269, 0.03445185720920563, 0.04..."


In [16]:
output_normalized_parquet_path = '/content/drive/MyDrive/Tesis/books_with_embeddings_norm.parquet'
df.to_parquet(output_normalized_parquet_path, index=False)
print(f"DataFrame con embeddings normalizados guardado en {output_normalized_parquet_path}")

DataFrame con embeddings normalizados guardado en /content/drive/MyDrive/Tesis/books_with_embeddings_norm.parquet
